In [ ]:
import os
import numpy as np
import mujoco
from mujoco import mjx
import jax
import myosuite
from myosuite.utils import gym
import jax.numpy as jp
from brax.io import html, mjcf
from stable_baselines3 import PPO
from IPython.display import HTML, display
import matplotlib.pyplot as plt


HERE = os.getcwd()
MODEL_PATH = os.path.join(HERE, "myosuite/envs/myo/assets/arm/myoarm_bionic_bimanual_mjx.xml")
PPO_MODEL_PATH = os.path.join(HERE, "myosuite/agents/baseline.zip")

try:
    mj_model = mujoco.MjModel.from_xml_path(MODEL_PATH)
    print("Model loaded successfully.")
except Exception as e:
    print(f"Failed to load model: {e}")
    # Optionally, you can print the stack trace for more details
    import traceback
    traceback.print_exc()

mj_data   = mujoco.MjData(mj_model)
renderer  = mujoco.Renderer(mj_model)
mjx_model = mjx.put_model(mj_model)
mjx_data  = mjx.put_data(mj_model, mj_data)

In [3]:
print('Installing mediapy:')
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
import mediapy as media
import matplotlib.pyplot as plt

scene_option = mujoco.MjvOption()
scene_option.flags[mujoco.mjtVisFlag.mjVIS_JOINT] = True

duration = 3.8  # (seconds)
framerate = 60  # (Hz)

rollout = []
mujoco.mj_resetData(mj_model, mj_data)
while mj_data.time < duration:
  mujoco.mj_step(mj_model, mj_data)
  if len(rollout) < mj_data.time * framerate:
    renderer.update_scene(mj_data, scene_option=scene_option)
    pixels = renderer.render()
    rollout.append(pixels)

# Simulate and display video.
media.show_video(rollout, fps=framerate)
media.save_video(rollout, fps=framerate, path="simulation.mp4")

Installing mediapy:


This browser does not support the video tag.


AttributeError: module 'mediapy' has no attribute 'save_video'

In [ ]:
html_content = html.render(mjx_model, rollout, camera="track")
display(HTML(html_content))
with open('output.html', 'w', encoding='utf-8') as file:
    file.write(html_content)